# Scenario 3 — A TransferUnit Driven Over MQTT (Ontology-Driven Wiring)

The **mock PLC → middleware → controller** loop of wayfinder map #24.

Where scenario 1 demonstrates operation coordination and scenario 2 direct workflow
invocation, this one demonstrates **ontology-driven wiring**. Nothing in this notebook names
a topic or a broker address. The middleware reads the seeded TransferUnit out of the
knowledge graph, works out that four of its properties are interface-accessible parameters,
and builds six MQTT connectors pointed at the addresses the graph gave it.

The device end is deliberately dumb: `MockTransferUnit` speaks MQTT and knows nothing about
the graph, the ontology or the middleware. That asymmetry is what the scenario exists to
show — all the semantics live on the middleware side.

**Prerequisites**: the `GRAPHDB_*` environment variables, and an MQTT broker on
`127.0.0.1:1883`. If nothing is listening there, the next cell starts a pure-Python one.

A plain-Python equivalent, better suited to a debugger, is `scenario3_transferunit.py`.

In [1]:
import asyncio
import json
import os
import sys

sys.path.insert(0, ".")

from graph_db_interface import GraphDB
from kapps_ogm import OGM
from aas_middleware.middleware.sync.synced_connector import SyncDirection

from kapps_semantic_middleware import SemanticMiddleware
from kapps_semantic_middleware.connectors.semantic import default_registry
from kapps_semantic_middleware.projection import carries_southbound
from kapps_semantic_middleware.vocabulary import INF

import seed
from mock_transferunit import MockTransferUnit
from scenario3_transferunit import (
    BROKER_HOST, BROKER_PORT, SETPOINT,
    transfer_unit_view, _broker_is_running, _registration,
)

db = GraphDB.from_env()
print("Connected to GraphDB repository:", os.getenv("GRAPHDB_REPOSITORY"))

broker = None
if not _broker_is_running():
    from amqtt.broker import Broker
    broker = Broker({
        "listeners": {"default": {"type": "tcp", "bind": f"{BROKER_HOST}:{BROKER_PORT}"}},
        "sys_interval": 0,
        "auth": {"allow-anonymous": True},
        "topic-check": {"enabled": False},
    })
    await broker.start()
    print(f"Started an in-process MQTT broker on {BROKER_HOST}:{BROKER_PORT}")
else:
    print(f"Using the MQTT broker already on {BROKER_HOST}:{BROKER_PORT}")

INFO:KafkaManager:KafkaManager initialized


INFO:GraphDB:Using GraphDB repository 'Tests' as user 'etienneh'.


/tmp/ipykernel_150133/509024345.py:30: DeprecationWarning: Loading plugins from EntryPoints is deprecated and will be removed in a future version. Use `plugins` section of config instead.
  broker = Broker({


INFO:transitions.core:Executed callback '<bound method Broker._log_state_change of <amqtt.broker.Broker object at 0x75fdf83e99a0>>'


INFO:transitions.core:Finished processing state new exit callbacks.


...

INFO:amqtt.broker:Listener 'default' bind to 127.0.0.1:1883 (max_connections=0)


INFO:transitions.core:Finished processing state starting exit callbacks.


INFO:transitions.core:Finished processing state started enter callbacks.


INFO:amqtt.broker:Starting session expiration monitor.


Connected to GraphDB repository: Tests
Started an in-process MQTT broker on 127.0.0.1:1883


## Step 1 — Seed a Clean Repository

Every scenario clears its repository and creates its own starting data, so the notebook is a
complete, reproducible specification of its own prerequisites (ADR 0010).

The TransferUnit, its two conveyor belts and its two light barriers are created **through the
OGM**, the same validated write path a running middleware uses (root ADR 0008). The ontology
file is classes only; the instances are built here. Note the MQTT connection metadata is
written as part of that same `ogm.create` call — until `kapps_ogm#7` landed it had to be
patched in afterwards with a raw SPARQL `INSERT`.

In [2]:
seed.seed_scenario3(db, OGM(db=db))

rows = db.query(
    f"SELECT ?n ?t FROM <http://www.ontotext.com/explicit> WHERE {{ ?n <{INF.hasMQTTTopic}> ?t }}",
    convert_bindings=True,
)["results"]["bindings"]

print(f"Topics recorded in the graph: {len(rows)}")
for r in rows:
    print("  ", str(r["t"]))

Topics recorded in the graph: 4
   TransferUnit1/ConveyorBelt/left/speed
   TransferUnit1/ConveyorBelt/right/speed
   TransferUnit1/LightBarrier/front/occupied
   TransferUnit1/LightBarrier/back/occupied


## Step 2 — Construct the Middleware

Passing a `class_scope` is what asks for the parameters under it to be wired. It is the
**consumer's view**, rooted at this resource and configured here in embedding code rather
than in the ontology (ADR 0018) — two levels deep, because a TransferUnit's parameters hang
off its belts and barriers, which the ontology cannot guess.

Wiring happens in the **constructor**, not on startup. `lifespan` calls `connect()` on
everything in the connection registry *before* running `on_start_up` callbacks, and
`initiate_sync` never calls it — so a connector registered later never connects, its listener
never starts, and `receive()` blocks forever while outbound limps on through `consume()`'s
reconnect. Silent, one-directional failure (ADR 0023).

In [3]:
unit = SemanticMiddleware(
    mode="resource",
    resource_iri=seed.TRANSFER_UNIT_1,
    service_class=f"{seed.TU_NS}TransferUnitService",
    ogm=OGM(db=db),
    host="127.0.0.1",
    port=8996,
    class_scope=transfer_unit_view(),
    # The controller flavour. A monitor passes TO_PERSISTENCE; an inspector passes
    # autoregister_connectors=False (ADR 0022).
    connector_sync_direction=SyncDirection.BIDIRECTIONAL,
)

print("Binding descriptors in the registry:", len(default_registry))
print("Connectors registered before startup:", bool(unit.connection_registry.connections))

INFO:kapps_semantic_middleware.middleware:Wired 6 connector(s) across 4 parameter(s) on https://www.sfb1574.kit.edu/ontologies/TransferUnitInstances#TransferUnit1


Binding descriptors in the registry: 1
Connectors registered before startup: True


## Step 3 — What Recognition Found

Recognition matches on the **interface property**, not on `rdf:type`. A parameter node has no
named type of its own — only anonymous restriction nodes, which exist by inference and never
survive an explicit-graph fetch. The property hierarchy is what survives a round trip
(ADR 0020), so `tu:hasConveyorSpeed ⊑ inf:isInterfaceAccessibleMQTTParameter` is the match.

**4 parameters → 4 bindings → 6 connectors → 6 topics.** A settable parameter needs *two*
connectors: `MqttClientConnector` publishes where it subscribed, so one instance cannot serve
both a read topic and a distinct set topic. Both bind to one `ConnectionInfo` and differ only
in direction (ADR 0023).

In [4]:
plan = unit._wiring

print(f"Parameters recognised: {len(plan.bindings)}")
for b in plan.bindings:
    print(f"  {str(b.resource_iri).split('#')[-1]:22}"
          f" {str(b.parameter_property).split('#')[-1]:18} accessMode={b.access_mode}")

print(f"\nConnectors built: {len(plan.registrations)}")
for _, r in plan.registrations:
    d = "read " if r.sync_direction is SyncDirection.TO_PERSISTENCE else "write"
    print(f"  {d}  {r.connector.topic}")

Parameters recognised: 4
  LightBarrier1_front    isOccupied         accessMode=read
  LightBarrier1_back     isOccupied         accessMode=read
  ConveyorBelt1_right    hasConveyorSpeed   accessMode=readwrite
  ConveyorBelt1_left     hasConveyorSpeed   accessMode=readwrite

Connectors built: 6
  read   TransferUnit1/LightBarrier/front/occupied
  read   TransferUnit1/LightBarrier/back/occupied
  read   TransferUnit1/ConveyorBelt/right/speed
  write  TransferUnit1/ConveyorBelt/right/speed_set
  read   TransferUnit1/ConveyorBelt/left/speed
  write  TransferUnit1/ConveyorBelt/left/speed_set


## Step 4 — The Northbound Projection

The served payload carries the value, the unit and the access mode — and nothing that would
let a peer reach the device directly.

This is not a filter applied to the data. The southbound properties are removed from the
**ClassSpec** before anything is fetched, so the northbound model has no field that could
carry a broker address (ADR 0028). Which properties are southbound is never named in the
core: the registry takes the union of every registered binding's `connection_metadata`, so a
domain expert's own connector is projected out for free.

The prune runs for **every flavour**, including an inspector that wires nothing — gating it
would make the least-privileged instance the one that leaks.

In [5]:
node = unit.ogm.fetch(instance_iri=seed.TRANSFER_UNIT_1, **plan.northbound_fetch_kwargs())
served = node.instance.model_dump()

belt = served[seed.TU_HAS_CONVEYOR_BELT.lined][0]
parameter = belt[seed.TU_HAS_CONVEYOR_SPEED.lined][0]

print("A belt's speed parameter, as a peer would receive it:")
for field, value in parameter.items():
    print(f"  {field.split('_h_')[-1]:16} = {value}")

leaks = carries_southbound(served, plan.southbound_properties)
print("\nConnection metadata in the served payload:", leaks or "none")
assert not leaks and seed.MQTT_BROKER_IP not in str(served)

A belt's speed parameter, as a peer would receive it:
  hasValue         = []
  hasUnit          = ['m/s']
  accessMode       = ['readwrite']

Connection metadata in the served payload: none


## Step 5 — Start the Mock PLC

The edge device. It publishes four values and subscribes to two setpoints, and it knows
nothing about any of the above. Publishing is periodic as well as on-change, so a middleware
that starts *after* the device still converges — `MqttClientConnector` holds the latest
message from its subscription, and has nothing to hold until one arrives.

In [6]:
plc = MockTransferUnit(broker=BROKER_HOST, port=BROKER_PORT, publish_interval=0.2)
await plc.start()

print("Publishes:", *plc.published_topics, sep="\n  ")
print("Subscribes:", *plc.subscribed_topics, sep="\n  ")

INFO:mock_transferunit:MockTransferUnit TransferUnit1 up on 127.0.0.1:1883 — publishing 4, subscribed to 2


Publishes:
  TransferUnit1/ConveyorBelt/left/speed
  TransferUnit1/ConveyorBelt/right/speed
  TransferUnit1/LightBarrier/front/occupied
  TransferUnit1/LightBarrier/back/occupied
Subscribes:
  TransferUnit1/ConveyorBelt/left/speed_set
  TransferUnit1/ConveyorBelt/right/speed_set


## Step 6 — A Live Value Flows Device → Middleware

The connector the graph built receives the device's reading.

The formatter then rebuilds the **whole parameter node**, not just the value.
`update_persistence_with_value` does `setattr(contained_model, field_id, value)` — replacing
the node wholesale — and `Formatter.deserialize` sees only the payload, with no access to the
current value. A bare scalar would therefore blank the unit and the access mode in the very
model that is served over REST.

In [7]:
read = _registration(unit, "ConveyorBelt/left/speed", SyncDirection.TO_PERSISTENCE)
await read.connector.connect()

value = await asyncio.wait_for(read.connector.queue.get(), timeout=10.0)
print(f"Received on {read.connector.topic}: {value}")

[node] = read.formatter.deserialize(value)
print("  value     ", getattr(node, INF.hasValue.lined))
print("  unit kept ", getattr(node, seed.TU_HAS_UNIT.lined))
print("  mode kept ", getattr(node, INF.accessMode.lined))

Received on TransferUnit1/ConveyorBelt/left/speed: 0.0
  value      [0.0]
  unit kept  ['m/s']
  mode kept  ['readwrite']


## Step 7 — A Setpoint Flows Middleware → Device

The write connector publishes to the parameter's `inf:hasMQTTSetTopic`, the PLC applies it,
and reports the new speed back on its read topic. That is the loop closing.

The payload path here is exactly what the sync machinery does: persistence value →
`formatter.serialize` → `connector.consume`.

In [8]:
write = _registration(unit, "ConveyorBelt/left/speed_set", SyncDirection.FROM_PERSISTENCE)
await write.connector.connect()

print("Belt speed before:", plc.speeds["left"])

[n] = write.formatter.deserialize(SETPOINT)
payload = write.formatter.serialize([n])
print(f"Publishing {json.loads(payload)} to {write.connector.topic}")
await write.connector.consume(payload)
await plc.wait_for_setpoint(timeout=10.0)

print("Belt speed after: ", plc.speeds["left"])

async def until_setpoint_reported():
    while True:
        if await read.connector.queue.get() == SETPOINT:
            return SETPOINT

reported = await asyncio.wait_for(until_setpoint_reported(), timeout=10.0)
print(f"Device reported it back on {read.connector.topic}: {reported}")

Belt speed before: 0.0
Publishing 2.5 to TransferUnit1/ConveyorBelt/left/speed_set
Belt speed after:  2.5
Device reported it back on TransferUnit1/ConveyorBelt/left/speed: 2.5


## Step 8 — A Read-Only Parameter

A light barrier is an occupancy sensor. Its `inf:accessMode` is `read`, so the middleware
builds **no write connector** for it — a controller cannot drive a sensor, structurally
rather than by convention.

Direction is the most restrictive of the parameter's access mode and the instance's flavour,
and neither may widen the other. An absent or unrecognised access mode yields read-only, so a
parameter is never writable by accident of omission (ADR 0023).

In [9]:
writable = [r for _, r in plan.registrations
            if "occupied" in r.connector.topic
            and r.sync_direction is SyncDirection.FROM_PERSISTENCE]
print("Write connectors built for the barriers:", len(writable))
assert not writable

barrier = _registration(unit, "LightBarrier/front/occupied", SyncDirection.TO_PERSISTENCE)
await barrier.connector.connect()
await asyncio.sleep(0.3)
await plc.set_occupied("front", True)

async def until_occupied():
    while True:
        if await barrier.connector.queue.get() is True:
            return True

await asyncio.wait_for(until_occupied(), timeout=10.0)
print(f"Barrier tripped, observed on {barrier.connector.topic}: True")

Write connectors built for the barriers: 0


Barrier tripped, observed on TransferUnit1/LightBarrier/front/occupied: True


## Step 9 — The Live Value is Never Persisted

Scenario 3 is a **locator** (ADR 0024): the graph records *where* a value lives — its unit,
access mode, topic and broker — and never the value itself. The live value exists only in the
datamodel and over REST.

A parameter that has not been observed yet simply has no value triple, which under the Open
World Assumption is an ordinary state rather than an error. A domain whose data changes
slowly may instead *commit* its values; the middleware is agnostic between the two patterns.

In [10]:
rows = db.query(
    f"SELECT ?v FROM <http://www.ontotext.com/explicit> WHERE {{ ?n <{INF.hasValue}> ?v }}",
    convert_bindings=True,
)["results"]["bindings"]

print("inf:hasValue literals in the graph:", len(rows))
assert not rows, "the live value must not be written to the graph"

inf:hasValue literals in the graph: 0


## Step 10 — Shutdown

In [11]:
for r in (read, write, barrier):
    await r.connector.disconnect()
await plc.stop()
if broker is not None:
    await broker.shutdown()

print("Scenario 3 complete.")

INFO:amqtt.broker:Shutting down broker...


INFO:transitions.core:Finished processing state started exit callbacks.


INFO:transitions.core:Finished processing state stopping enter callbacks.


INFO:amqtt.broker:Broker closed


INFO:transitions.core:Finished processing state stopping exit callbacks.


INFO:transitions.core:Finished processing state stopped enter callbacks.


Scenario 3 complete.
